In [ ]:
# transformer nochmal wiederholen und das allgemeine prinzip verstehen

In [ ]:
# daten laden und verstehen

import wget, os, gzip, pickle, random, re, sys, importlib, tqdm, math, os, gzip, re, string

from tqdm import trange
from collections import Counter

import torch

import random
from random import choice

IMDB_URL = 'http://dlvu.github.io/data/imdb.{}.pkl.gz'
IMDB_FILE = 'imdb.{}.pkl.gz'
WP_DATA = 'https://codeberg.org/pbm/former/raw/branch/master/data/enwik8.gz'

PAD, START, END, UNK = '.pad', '.start', '.end', '.unk'

SENT = '_s'
TOY = {
    '_s': ['_s _adv', '_np _vp', '_np _vp _prep _np', '_np _vp ( _prep _np )', '_np _vp _con _s','_np _vp ( _con _s )'],
    '_adv': ['briefly', 'quickly', 'impatiently'],
    '_np': ['a _noun', 'the _noun', 'a _adj _noun', 'the _adj _noun'],
    '_prep': ['on', 'with', 'to', 'for', 'at'],
    '_con': ['while', 'but'],
    '_noun': ['mouse', 'bunny', 'cat', 'dog', 'man', 'woman', 'person', 'bear', 'koala', 'judge', 'businessman',
        'businesswoman', 'lawyer', 'teacher', 'engineer'],
    '_vp': ['walked', 'walks', 'ran', 'runs', 'goes', 'went', 'hiked'],
    '_adj': ['short', 'quick', 'busy', 'nice', 'gorgeous', 'spectacular', 'reluctant', 'systematic', 'willowy', 'engaged', 'synthetic']
}

PRINTABLE = set(ord(c) for c in (string.digits + string.ascii_letters + string.punctuation + string.whitespace))

def cas(i):
    """
    Character-as-string. Filters out the ascii codes that aren't safe to print.
    :return:
    """
    assert i >= 0 and i < 256
    return '□' if i not in PRINTABLE else str(chr(i))

def t(blist):
    return torch.tensor([int(b) for b in blist], dtype=torch.uint8)

def gen_sentence(sent=SENT, g=TOY):

    symb = '_[a-z]*'

    while True:

        match = re.search(symb, sent)
        if match is None:
            return sent

        s = match.span()
        sent = sent[:s[0]] + random.choice(g[sent[s[0]:s[1]]]) + sent[s[1]:]

def load_toy(ntrain=100_000, ntest=20_000, to_torch=True, final=False, seed=0):
    """
    Generates language from a toy grammar.
    :param ntrain:
    :param ntest:
    :param to_torch: Whether to return torch tensors (if false, returns python lists)
    :param final: Whether to return the test set or the validation set (True for test)
    :return:
    """

    random.seed(seed)

    train, test = '', ''
    while len(train) < ntrain:
        train += gen_sentence() + ' . '

    random.seed(seed if final else seed + 1)
    # -- change the seed so we get different test/val sets depending on `final`

    while len(test) < ntest:
        test += gen_sentence() + ' . '

    ctr = Counter(train + test)
    i2t = [PAD, START, END, UNK] + [t for t, _ in ctr.most_common()]
    t2i = { w : i for  i, w in enumerate(i2t)}

    train = [t2i[t] for t in train]
    test  = [t2i[t] for t in test]
    
    if to_torch:
        return (t(train), t(test)), (i2t, t2i) # Torch vectors (this takes a few seconds)

    return (train, test), (i2t, t2i)

def load_wp(fname='enwik8.gz', split=(90, 5, 5), to_torch=True, final=False):
    """
    Load the enwik8 dataset from the Hutter challenge as a list or vector of bytes.
    :param fname: Filename for the downloaded data.
    :param split: Percentages for the train/val/test split.
    :param to_torch: Whether to return torch tensors (True) or python lists (False)
    :param final: If False, returns train/val if True returns train/test with the validation
    data added to the training data.
    :return:
    """

    if not os.path.exists(fname):
        # If it doesn't exist, download it
        print('Downloading')
        wget.download(WP_DATA, out=fname)
        
    with gzip.open(fname, 'r') if fname.endswith('.gz') else open(fname, 'rb') as file:

        all = file.read()
        ctr = Counter(all)

        i2t = {token : cas(token) for token, freq in ctr.most_common()}
        t2i = {w : i for i, w in enumerate(i2t)}

        split = tuple(s/sum(split) for s in split)
        split = tuple(int(s * len(all)) for s in split)

        train, val, test = all[:split[0]], all[split[0]:split[0]+split[1]], all[split[0]+split[1]:]

        if final:
            train = train + val
            wh = test
        else:
            wh = val

        if to_torch:
            return (t(train), t(wh)), (i2t, t2i)

        return (train, wh), (i2t, t2i)

def load_xor(ntrain=25_000, ntest=25_000, seed=0):

    random.seed(seed)

    i2w = [PAD, START, END, UNK, 'true', 'false'] #
    w2i = {w : i for i, w in enumerate(i2w)}

    dataset, labels = [], []
    for _ in range(ntrain + ntest):
        sentence = [
            choice((i2w[4], i2w[5])),
            choice((i2w[4], i2w[5]))
        ]

        f1, f2 = (sentence[0] == i2w[4]), (sentence[1] == i2w[4]) # true: very/great false: not/terrible
        # -- these words are the only meaningful features
        label = 0 if f1 != f2 else 1

        dataset.append([w2i[word] for word in sentence])
        labels.append(label)

    return \
        (dataset[:ntrain], labels[:ntrain]), \
        (dataset[ntrain:], labels[ntrain:]), \
        (i2w, w2i), 2

def load_imdb_synth(ntrain=25_000, ntest=25_000, seed=0):
    """
    Synthetic IMDb dataset
    :param seed:
    :param voc:
    :return:
    """

    random.seed(seed)

    adjectives = ['classic', 'silent', 'modern', 'vintage', 'independent', 'foreign', 'animated', 'documentary',
    'epic', 'dramatic', 'romantic', 'comic', 'thrilling', 'mysterious', 'gritty', 'stylized', 'iconic', 'acclaimed',
    'popular', 'forgettable', 'unreleased', 'awardwinning', 'blockbuster', 'lowbudget', 'highbudget', 'experimental',
    'mainstream', 'cult', 'notable', 'original']
    nouns = ['movie', 'film', 'motion-picture', 'feature', 'featurette', 'picture', 'flick', 'cinema', 'screenplay',
    'blockbuster', 'talkie', 'silent', 'biopic', 'short', 'docudrama', 'documentary', 'animation', 'cartoon',
    'anime', 'telefilm', 'miniseries', 'drama', 'comedy', 'thriller', 'western', 'musical', 'noir']
    verbs = ['was', 'is', 'became', 'becomes', 'seemed', 'seems']

    i2w = [PAD, START, END, UNK, 'this', 'not', 'very', 'great','terrible'] + verbs + adjectives + nouns
    w2i = {w : i for i, w in enumerate(i2w)}

    dataset, labels = [], []
    for _ in range(ntrain + ntest):
        sentence = [
            i2w[4], # this
            choice(adjectives), # old
            choice(nouns), # movie
            choice(verbs), # was
            choice((i2w[5], i2w[6])),
            choice((i2w[7], i2w[8]))
        ]

        f1, f2 = (sentence[4] == i2w[6]), (sentence[5] == i2w[7]) # true: very/great false: not/terrible
        # -- these words are the only meaningful features
        label = 0 if f1 != f2 else 1

        dataset.append([w2i[word] for word in sentence])
        labels.append(label)

    return \
        (dataset[:ntrain], labels[:ntrain]), \
        (dataset[ntrain:], labels[ntrain:]), \
        (i2w, w2i), 2

def load_imdb(final=False, val=5000, seed=0, voc=None, char=False):

    cst = 'char' if char else 'word'

    imdb_url = IMDB_URL.format(cst)
    imdb_file = IMDB_FILE.format(cst)

    if not os.path.exists(imdb_file):
        wget.download(imdb_url)

    with gzip.open(imdb_file) as file:
        sequences, labels, i2w, w2i = pickle.load(file)

    if voc is not None and voc < len(i2w):
        nw_sequences = {}

        i2w = i2w[:voc]
        w2i = {w: i for i, w in enumerate(i2w)}

        mx, unk = voc, w2i['.unk']
        for key, seqs in sequences.items():
            nw_sequences[key] = []
            for seq in seqs:
                seq = [s if s < mx else unk for s in seq]
                nw_sequences[key].append(seq)

        sequences = nw_sequences

    if final:
        return (sequences['train'], labels['train']), (sequences['test'], labels['test']), (i2w, w2i), 2

    # Make a validation split
    random.seed(seed)

    x_train, y_train = [], []
    x_val, y_val = [], []

    val_ind = set( random.sample(range(len(sequences['train'])), k=val) )
    for i, (s, l) in enumerate(zip(sequences['train'], labels['train'])):
        if i in val_ind:
            x_val.append(s)
            y_val.append(l)
        else:
            x_train.append(s)
            y_train.append(l)

    return (x_train, y_train), \
           (x_val, y_val), \
           (i2w, w2i), 2

In [ ]:
# utils.py

def batchify_rand(x, bsz, L):
  N = x.size(0)
  rand_start_indices = torch.randint(0, (N-L), (bsz,)).unsqueeze(1) # (bsz, 1)
  off_sets = torch.arange(L+1).unsqueeze(0) # (1,L+1)
  indices = rand_start_indices + off_sets # (bsz, 1) + (1,L+1) -> broadcasting
  batch = x[indices].long() # slice out each seq of batch in parallel and conver to int
  return batch

def save_to_json():
  pass



tensor([[ 4,  6,  4,  ...,  4, 26,  4],
        [ 4, 17,  6,  ...,  8, 20, 11],
        [11, 19,  5,  ..., 11, 17, 19],
        ...,
        [ 8, 20, 11,  ..., 15, 21,  4],
        [ 9, 11,  8,  ..., 19, 15, 21],
        [ 4, 16, 13,  ...,  6,  8,  4]])


In [ ]:
# model.py

import torch
import torch.nn as nn
import torch.functional as F


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, num_classes, vocab_size, num_heads, max_seq_length, embedding_dim = 300):
        super().__init__()

        # Define the three specific layers
        self.linear_K = nn.Linear(embedding_dim, embedding_dim)
        self.linear_Q = nn.Linear(embedding_dim, embedding_dim)
        self.linear_V = nn.Linear(embedding_dim, embedding_dim)

        self.linear_O = nn.Linear(embedding_dim, embedding_dim)
        
        self.num_heads = num_heads
        self.embedding_dim = embedding_dim


    def forward(self, combined_embed):

        K = self.linear_K(combined_embed) # (B,T,emb)
        Q = self.linear_Q(combined_embed) # (B,T,emb)
        V = self.linear_V(combined_embed) # (B,T,emb)

        hemb = self.embedding_dim // self.num_heads

        K = torch.reshape(K, (K.size(0), K.size(1), self.num_heads, hemb)) # (B,T, k, hemb)
        Q = torch.reshape(Q, (Q.size(0), Q.size(1), self.num_heads, hemb)) # (B,T, k, hemb)
        V = torch.reshape(V, (V.size(0), V.size(1), self.num_heads, hemb)) # (B,T, k, hemb)

        K = K.permute(0,2,1,3) # (B, k, T, hemb)
        Q = Q.permute(0,2,1,3) # (B, k, T, hemb)
        V = V.permute(0,2,1,3) # (B, k, T, hemb)

        K_Q = K @ Q.transpose(-1, -2) # (B, k, T, T)

        K_Q_scaled = K_Q / math.sqrt(hemb) # (B, k, T, T)

        W = F.softmax(K_Q_scaled, dim=-1) # (B, k, T, T)

        Z = W @ V # (B, k, T, hemb)

        Z = Z.permute(0,2,1,3) # (B, T, k, hemb)

        Z = torch.reshape(Z, (Z.size(0), Z.size(1), self.embedding_dim)) # (B, T, emb)

        O = self.linear_O(Z) # (B, T, emb)

        return O

class TransformerBlock(nn.Module):
    def __init__(self, num_classes, vocab_size, num_heads, max_seq_length, embedding_dim = 300, dropout = 0.1):
        super().__init__()
        self.selfatt = MultiHeadSelfAttention(num_classes, vocab_size, num_heads, max_seq_length)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.ff = nn.Sequential(
            nn.Linear(embedding_dim, 4*embedding_dim),
            nn.ReLU(),
            nn.Linear(4*embedding_dim, embedding_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, combined_embed):
        # Sub-layer 1: LN + SelfAtt + DropOut + Resid
        combined_embed_norm = self.ln1(combined_embed)  # (B, T, emb)
        att_out = self.selfatt(combined_embed_norm)     # (B, T, emb)
        att_out = self.dropout(att_out)                 # (B, T, emb)
        residual_added = att_out + combined_embed       # (B, T, emb)

        # Sub-layer 2: LN + FF + DropOut + Resid
        ln_2 = self.ln2(residual_added)                 # (B, T, emb)
        ff_out = self.ff(ln_2)                          # (B, T, emb)
        ff_out = self.dropout(ff_out)                   # (B, T, emb)
        out_resid2 = ff_out + residual_added            # (B, T, emb)

        return out_resid2

class Transformer(nn.Module):
    def __init__(self, num_classes, vocab_size, num_heads, num_Tblocks, max_seq_length = 256, pool = 'mean', embedding_dim = 300, dropout = 0.1):
        super().__init__()
        self.blocks = nn.ModuleList([TransformerBlock(num_classes, vocab_size, num_heads, max_seq_length) for _ in range(num_Tblocks)])
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pos_embedding = nn.Embedding(max_seq_length, embedding_dim)
        self.Linear = nn.Linear(embedding_dim, num_classes)
        self.pool = pool
        self.max_seq_length = max_seq_length

    def forward(self, x):
        B, T = x.size() # B = batch dimension, T = time dimension

        #Trim sequences that are too long
        if x.size(1) > self.max_seq_length:
            x = x[:,-self.max_seq_length:]

        # Input and position embeddings
        input_embeddings = self.embedding(x) # (B, T, emb)
        time_dimension = torch.arange(T, device=device) # (T,)
        pos_embeddings = self.pos_embedding(time_dimension) # (T, emb)

        final_embeddings = input_embeddings + pos_embeddings # (B, T, emb) + (T, emb) -> broadcasting

        # Apply transformer blocks
        out = final_embeddings # (B, T, emb)
        for block in self.blocks:
            out = block(out) # (B, T, emb)

        # Pooling
        if self.pool == "mean":
            pooled = out.mean(dim=1) # (B, emb)
        elif self.pool == "max":
            pooled = out.max(dim=1).values  # (B, emb)
        elif self.pool == "first_elem":
            pooled = out[:,0,:]  # (B, emb)
        else:
            raise ValueError("UNKNOWN TYPE OF POOLING!")

        # Linear Layer Classifier
        logits = self.Linear(pooled) # (B, num_class)
        return logits

In [ ]:
# train.py

# imports

# dataclasses für storage von a) parametern und b) results

def train():
    pass


def validation():
    pass


def test():
    pass

'''
def run_from_cfg(cfg):
    pass
'''

def main():
    (train, test), (i2c, c2i) = load_toy(final=False)


if __name__ == "__main__":
    main()

In [ ]:
# run experiments

In [ ]:
# run experiments

In [ ]:
# run experiments

### 2.) Wikipedia Dataset

In [ ]:
# load wikipedia data set

'''
This dataset contains 100mb of text from wikipedia (taken from the first Hutter prize)
encoded as a sequence of bytes. That is, we have 256 tokens.
These mostly correspond directly to ASCII characters, but special characters are encoded in
multiple bytes.
Use the same model as in the previous part, but increase the hidden size, the embedding
dimension and possibly the number of layers.
'''